# Crawler + NER : Acquisition de données musicales

Ce notebook fait deux choses :
1. **Crawler** : scraper Wikipedia et MusicBrainz pour collecter des textes sur nos artistes
2. **NER** : extraire automatiquement des entités (artistes, genres, labels) depuis ces textes



### Installation

In [1]:
!conda install -c conda-forge spacy -y


3 channel Terms of Service accepted
Channels:
 - conda-forge
 - defaults
Platform: win-64
Solving environment: / - \ | / done

## Package Plan ##

  environment location: C:\Users\sandy\anaconda3\envs\rstudio

  added / updated specs:
    - spacy


The following packages will be UPDATED:

  spacy                                3.2.0-py37h457096e_0 --> 3.3.1-py37h81562fc_0 



Preparing transaction: \ done
Verifying transaction: / - \ | / - \ | / done
Executing transaction: \ | / - \ | / - \ | / done




==> WARNING: A newer version of conda exists. <==
    current version: 25.5.1
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c defaults conda




In [2]:
!conda install -c conda-forge spacy=3.2.0 -y

3 channel Terms of Service accepted
Channels:
 - conda-forge
 - defaults
Platform: win-64
Solving environment: / - \ | / done

## Package Plan ##

  environment location: C:\Users\sandy\anaconda3\envs\rstudio

  added / updated specs:
    - spacy=3.2.0


The following NEW packages will be INSTALLED:

  requests           conda-forge/noarch::requests-2.32.2-pyhd8ed1ab_0 
  urllib3            conda-forge/noarch::urllib3-2.2.1-pyhd8ed1ab_0 

The following packages will be DOWNGRADED:

  spacy                                3.3.1-py37h81562fc_0 --> 3.2.0-py37h457096e_0 



Preparing transaction: \ done
Verifying transaction: / - \ | / - \ | / done
Executing transaction: \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | done




==> WARNING: A newer version of conda exists. <==
    current version: 25.5.1
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c defaults conda




In [3]:
!pip install requests beautifulsoup4
print(" OK")

 OK


### Imports

In [4]:
import subprocess
subprocess.run(["pip", "install", "urllib3==1.26.18", "requests==2.31.0", "--quiet", "--force-reinstall"])
print("Fix OK")

Fix OK


In [5]:
import requests
import time
import json
import re
import spacy

CHEMIN = "C:/Users/sandy/OneDrive/Desktop/Cours A4/Web Datamining/TD4_Project/"
HEADERS = {"User-Agent": "MusicKG-StudentCrawler/1.0 (educational project)"}

# Chargement du modele spaCy
nlp = spacy.load("en_core_web_sm")

print("Imports OK")
print(f"spaCy version : {spacy.__version__}")

Imports OK
spaCy version : 3.8.7


### Liste des artistes a scraper

On scrape les memes artistes que notre KB pour enrichir les données.

In [6]:
ARTISTES = [
    "The Beatles", "Pink Floyd", "Led Zeppelin", "Radiohead", "David Bowie",
    "Daft Punk", "Stromae", "Celine Dion", "Edith Piaf", "Serge Gainsbourg",
    "Michael Jackson", "Bob Dylan", "Rolling Stones", "Queen", "Nirvana",
    "Beyonce", "Eminem", "Jay-Z", "Kanye West", "Adele",
    "Miles Davis", "Nina Simone", "Elvis Presley", "Bob Marley", "Aretha Franklin",
    "Kendrick Lamar", "Drake", "Rihanna", "Amy Winehouse", "Coldplay"
]

print(f"{len(ARTISTES)} artistes a scraper")

30 artistes a scraper


### Source 1 : Wikipedia

On utilise l'API Wikipedia (pas de scraping HTML brut) qui est plus stable
et respecte mieux les conditions d'utilisation.
L'API Wikipedia autorise les requetes automatisees avec un User-Agent identifie.
On ajoute un delai de 1 seconde entre chaque requete.

In [7]:
def scraper_wikipedia(artiste: str) -> dict:
    """
    Recupere le resume Wikipedia d'un artiste via l'API officielle.
    L'API Wikipedia est publique et autorise les requetes automatisees.
    """
    url = "https://en.wikipedia.org/api/rest_v1/page/summary/" + artiste.replace(" ", "_")
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        if r.status_code == 200:
            data = r.json()
            return {
                "artiste":  artiste,
                "source":   "wikipedia",
                "url":      data.get("content_urls", {}).get("desktop", {}).get("page", ""),
                "titre":    data.get("title", ""),
                "texte":    data.get("extract", ""),
                "statut":   "ok"
            }
        else:
            return {"artiste": artiste, "source": "wikipedia", "statut": f"erreur_{r.status_code}", "texte": ""}
    except Exception as e:
        return {"artiste": artiste, "source": "wikipedia", "statut": f"erreur_{str(e)}", "texte": ""}

# Test sur un artiste
test = scraper_wikipedia("Daft Punk")
print(f"Statut : {test['statut']}")
print(f"Texte  : {test['texte'][:300]}...")

Statut : ok
Texte  : Daft Punk were a French electronic music duo formed in 1993 in Paris by Thomas Bangalter and Guy-Manuel de Homem-Christo. They achieved popularity in the late 1990s as part of the French house movement, combining house music, funk, disco, techno, rock and synth-pop. They are regarded as one of the m...


In [8]:
# Scraping Wikipedia pour tous les artistes
textes_wikipedia = []
nb_ok = 0
nb_err = 0

for i, artiste in enumerate(ARTISTES):
    resultat = scraper_wikipedia(artiste)
    textes_wikipedia.append(resultat)

    if resultat["statut"] == "ok" and resultat["texte"]:
        nb_ok += 1
        print(f"  [{i+1}/{len(ARTISTES)}] {artiste} -> OK ({len(resultat['texte'])} chars)")
    else:
        nb_err += 1
        print(f"  [{i+1}/{len(ARTISTES)}] {artiste} -> {resultat['statut']}")

    time.sleep(1)  # delai ethique obligatoire

print()
print(f"Wikipedia : {nb_ok} OK, {nb_err} erreurs")

  [1/30] The Beatles -> OK (806 chars)
  [2/30] Pink Floyd -> OK (435 chars)
  [3/30] Led Zeppelin -> OK (511 chars)
  [4/30] Radiohead -> OK (397 chars)
  [5/30] David Bowie -> OK (356 chars)
  [6/30] Daft Punk -> OK (347 chars)
  [7/30] Stromae -> OK (508 chars)
  [8/30] Celine Dion -> OK (450 chars)
  [9/30] Edith Piaf -> OK (273 chars)
  [10/30] Serge Gainsbourg -> OK (598 chars)
  [11/30] Michael Jackson -> OK (591 chars)
  [12/30] Bob Dylan -> OK (590 chars)
  [13/30] Rolling Stones -> OK (737 chars)
  [14/30] Queen -> OK (151 chars)
  [15/30] Nirvana -> OK (229 chars)
  [16/30] Beyonce -> OK (369 chars)
  [17/30] Eminem -> OK (651 chars)
  [18/30] Jay-Z -> OK (446 chars)
  [19/30] Kanye West -> OK (461 chars)
  [20/30] Adele -> OK (303 chars)
  [21/30] Miles Davis -> OK (501 chars)
  [22/30] Nina Simone -> OK (421 chars)
  [23/30] Elvis Presley -> OK (406 chars)
  [24/30] Bob Marley -> OK (673 chars)
  [25/30] Aretha Franklin -> OK (185 chars)
  [26/30] Kendrick Lamar -> OK (329

### Source 2 : MusicBrainz

MusicBrainz est une base de données musicale ouverte.
Son API est publique et gratuite. Elle retourne des données structurées
sur les artistes : genres, pays, dates d'activité, labels.
Limite : 1 requete par seconde maximum.

In [9]:
def scraper_musicbrainz(artiste: str) -> dict:
    """
    Recupere les infos d'un artiste depuis l'API MusicBrainz.
    API publique, limite : 1 requete/seconde.
    """
    url = "https://musicbrainz.org/ws/2/artist/"
    params = {
        "query": f"artist:{artiste}",
        "fmt": "json",
        "limit": 1
    }
    try:
        r = requests.get(url, params=params, headers=HEADERS, timeout=10)
        if r.status_code == 200:
            data = r.json()
            artists = data.get("artists", [])
            if artists:
                a = artists[0]
                # Construction d'un texte descriptif depuis les donnees structurees
                tags  = [t["name"] for t in a.get("tags", [])[:5]]
                texte = f"{a.get('name', '')} is a {a.get('type', 'artist')} "
                texte += f"from {a.get('country', 'unknown')}. "
                if a.get("life-span", {}).get("begin"):
                    texte += f"Active since {a['life-span']['begin']}. "
                if tags:
                    texte += f"Genres: {', '.join(tags)}. "
                if a.get("disambiguation"):
                    texte += a["disambiguation"]
                return {
                    "artiste": artiste,
                    "source":  "musicbrainz",
                    "url":     f"https://musicbrainz.org/artist/{a.get('id', '')}",
                    "texte":   texte,
                    "tags":    tags,
                    "statut":  "ok"
                }
        return {"artiste": artiste, "source": "musicbrainz", "statut": f"erreur_{r.status_code}", "texte": ""}
    except Exception as e:
        return {"artiste": artiste, "source": "musicbrainz", "statut": str(e), "texte": ""}

# Test
test_mb = scraper_musicbrainz("Radiohead")
print(f"Statut : {test_mb['statut']}")
print(f"Texte  : {test_mb['texte']}")
print(f"Tags   : {test_mb.get('tags', [])}")

Statut : ok
Texte  : Radiohead is a Group from GB. Active since 1991. Genres: rock, electronic, post-rock, alternative rock, experimental. 
Tags   : ['rock', 'electronic', 'post-rock', 'alternative rock', 'experimental']


In [10]:
# Scraping MusicBrainz pour tous les artistes
textes_musicbrainz = []
nb_ok_mb = 0

for i, artiste in enumerate(ARTISTES):
    resultat = scraper_musicbrainz(artiste)
    textes_musicbrainz.append(resultat)

    if resultat["statut"] == "ok":
        nb_ok_mb += 1
        print(f"  [{i+1}/{len(ARTISTES)}] {artiste} -> OK")
    else:
        print(f"  [{i+1}/{len(ARTISTES)}] {artiste} -> {resultat['statut']}")

    time.sleep(1.5)  # MusicBrainz limite a 1 req/sec

print(f"\nMusicBrainz : {nb_ok_mb} OK")

  [1/30] The Beatles -> OK
  [2/30] Pink Floyd -> OK
  [3/30] Led Zeppelin -> OK
  [4/30] Radiohead -> OK
  [5/30] David Bowie -> OK
  [6/30] Daft Punk -> OK
  [7/30] Stromae -> OK
  [8/30] Celine Dion -> OK
  [9/30] Edith Piaf -> OK
  [10/30] Serge Gainsbourg -> OK
  [11/30] Michael Jackson -> OK
  [12/30] Bob Dylan -> OK
  [13/30] Rolling Stones -> OK
  [14/30] Queen -> OK
  [15/30] Nirvana -> OK
  [16/30] Beyonce -> OK
  [17/30] Eminem -> OK
  [18/30] Jay-Z -> OK
  [19/30] Kanye West -> OK
  [20/30] Adele -> OK
  [21/30] Miles Davis -> OK
  [22/30] Nina Simone -> OK
  [23/30] Elvis Presley -> OK
  [24/30] Bob Marley -> OK
  [25/30] Aretha Franklin -> OK
  [26/30] Kendrick Lamar -> OK
  [27/30] Drake -> OK
  [28/30] Rihanna -> OK
  [29/30] Amy Winehouse -> OK
  [30/30] Coldplay -> OK

MusicBrainz : 30 OK


### Nettoyage des textes

Avant le NER, on nettoie les textes :
- Suppression des caracteres speciaux
- Suppression des espaces multiples
- Suppression des parentheses et crochets
- Normalisation de la ponctuation

In [11]:
def nettoyer_texte(texte: str) -> str:
    """Nettoie un texte brut pour le NER."""
    if not texte:
        return ""
    # Suppression des references Wikipedia [1], [2]...
    texte = re.sub(r"\[\d+\]", "", texte)
    # Suppression des parentheses avec contenu court (ex: (born 1942))
    texte = re.sub(r"\([^)]{1,30}\)", "", texte)
    # Suppression des caracteres speciaux sauf ponctuation de base
    texte = re.sub(r"[^\w\s.,;:!?'-]", " ", texte)
    # Normalisation des espaces
    texte = re.sub(r"\s+", " ", texte).strip()
    return texte

# Application du nettoyage
tous_textes = []
for doc in textes_wikipedia + textes_musicbrainz:
    if doc["texte"]:
        texte_propre = nettoyer_texte(doc["texte"])
        tous_textes.append({
            "artiste": doc["artiste"],
            "source":  doc["source"],
            "texte":   texte_propre
        })

print(f"Textes nettoyes : {len(tous_textes)}")
print()
print("Exemple avant/apres nettoyage :")
ex_avant = textes_wikipedia[0]["texte"][:200]
ex_apres = nettoyer_texte(ex_avant)
print(f"Avant : {ex_avant}")
print(f"Apres : {ex_apres}")

Textes nettoyes : 60

Exemple avant/apres nettoyage :
Avant : The Beatles were an English rock band formed in Liverpool in 1960. The core lineup of the band comprised John Lennon, Paul McCartney, George Harrison and Ringo Starr. They are widely regarded as the m
Apres : The Beatles were an English rock band formed in Liverpool in 1960. The core lineup of the band comprised John Lennon, Paul McCartney, George Harrison and Ringo Starr. They are widely regarded as the m


### NER : extraction d'entites

spaCy identifie automatiquement les entites nommees dans le texte.
Les types qui nous interessent :
- `PERSON` : noms de personnes (artistes, membres...)
- `ORG` : organisations (labels, groupes...)
- `GPE` : lieux geographiques (pays, villes...)
- `DATE` : dates et periodes

In [12]:
def extraire_entites(texte: str) -> dict:
    """Extrait les entites nommees avec spaCy."""
    doc = nlp(texte[:5000])
    entites = {"PERSON": [], "ORG": [], "GPE": [], "DATE": []}
    for ent in doc.ents:
        if ent.label_ in entites:
            valeur = ent.text.strip()
            if valeur not in entites[ent.label_] and len(valeur) > 1:
                entites[ent.label_].append(valeur)
    return entites

# Application sur tous les textes
resultats_ner = []
for doc in tous_textes:
    entites = extraire_entites(doc["texte"])
    resultats_ner.append({
        "artiste": doc["artiste"],
        "source":  doc["source"],
        "entites": entites
    })

# Affichage
print("Exemples de resultats NER :")
print()
for r in resultats_ner[:4]:
    print(f"Artiste : {r['artiste']} ({r['source']})")
    for type_ent, liste in r["entites"].items():
        if liste:
            print(f"  {type_ent:<8} : {', '.join(liste[:5])}")
    print()

Exemples de resultats NER :

Artiste : The Beatles (wikipedia)
  PERSON   : John Lennon, Paul McCartney, George Harrison, Ringo Starr, Beatles
  GPE      : Liverpool
  DATE     : 1960, 1960s, 1950s

Artiste : Pink Floyd (wikipedia)
  PERSON   : Pink Floyd, Syd Barrett, Nick Mason, Roger Waters, Richard Wright
  GPE      : London
  DATE     : 1965, the end of 1967

Artiste : Led Zeppelin (wikipedia)
  PERSON   : Led Zeppelin, Robert Plant, Jimmy Page, John Paul Jones, John Bonham
  GPE      : London
  DATE     : 1968

Artiste : Radiohead (wikipedia)
  PERSON   : Thom Yorke, Jonny Greenwood, Colin Greenwood, Ed O'Brien, Philip Selway
  ORG      : Radiohead
  GPE      : Abingdon, Oxfordshire
  DATE     : 1985, 1994



### Analyse des cas ambigus

Le NER fait parfois des erreurs sur des noms ambigus.
On analyse 3 cas problematiques dans notre domaine musical.

In [13]:
print("ANALYSE DES CAS AMBIGUS")
print()

# Cas 1 : Queen
print("CAS 1 : 'Queen'")
texte_queen = "Queen is a British rock band formed in London in 1970. The band consists of Freddie Mercury, Brian May, Roger Taylor and John Deacon. The Queen of England visited the palace."
doc_queen = nlp(texte_queen)
print(f"Texte : {texte_queen}")
print("Entites detectees :")
for ent in doc_queen.ents:
    print(f"  '{ent.text}' -> {ent.label_}")
print("Probleme : 'Queen' peut designer le groupe musical ou la reine d'Angleterre.")
print("Solution : on verifie le contexte (presence de 'band', 'rock', noms de membres).")
print()

# Cas 2 : Nirvana
print("CAS 2 : 'Nirvana'")
texte_nirvana = "Nirvana is an American rock band from Seattle. In Buddhism, nirvana is the highest state of consciousness."
doc_nirvana = nlp(texte_nirvana)
print(f"Texte : {texte_nirvana}")
print("Entites detectees :")
for ent in doc_nirvana.ents:
    print(f"  '{ent.text}' -> {ent.label_}")
print("Probleme : 'Nirvana' est aussi un concept bouddhiste.")
print("Solution : on utilise le contexte geographique (Seattle) et le type (rock band).")
print()

# Cas 3 : Drake
print("CAS 3 : 'Drake'")
texte_drake = "Drake released his album in 2018. Sir Francis Drake was an English sea captain in the 16th century."
doc_drake = nlp(texte_drake)
print(f"Texte : {texte_drake}")
print("Entites detectees :")
for ent in doc_drake.ents:
    print(f"  '{ent.text}' -> {ent.label_}")
print("Probleme : 'Drake' peut etre le rappeur canadien ou l'explorateur Francis Drake.")
print("Solution : on verifie la periode temporelle et le contexte (album, music vs captain, century).")

ANALYSE DES CAS AMBIGUS

CAS 1 : 'Queen'
Texte : Queen is a British rock band formed in London in 1970. The band consists of Freddie Mercury, Brian May, Roger Taylor and John Deacon. The Queen of England visited the palace.
Entites detectees :
  'British' -> NORP
  'London' -> GPE
  '1970' -> DATE
  'Freddie Mercury' -> ORG
  'Brian May' -> PERSON
  'Roger Taylor' -> PERSON
  'John Deacon' -> PERSON
  'The Queen of England' -> WORK_OF_ART
Probleme : 'Queen' peut designer le groupe musical ou la reine d'Angleterre.
Solution : on verifie le contexte (presence de 'band', 'rock', noms de membres).

CAS 2 : 'Nirvana'
Texte : Nirvana is an American rock band from Seattle. In Buddhism, nirvana is the highest state of consciousness.
Entites detectees :
  'Nirvana' -> GPE
  'American' -> NORP
  'Seattle' -> GPE
  'Buddhism' -> GPE
  'nirvana' -> GPE
Probleme : 'Nirvana' est aussi un concept bouddhiste.
Solution : on utilise le contexte geographique (Seattle) et le type (rock band).

CAS 3 : 'Dr

### Statistiques finales

In [14]:
# Agregation de toutes les entites trouvees
toutes_personnes = set()
toutes_orgs      = set()
tous_lieux       = set()

for r in resultats_ner:
    toutes_personnes.update(r["entites"]["PERSON"])
    toutes_orgs.update(r["entites"]["ORG"])
    tous_lieux.update(r["entites"]["GPE"])

print("STATISTIQUES NER")
print(f"Textes analyses      : {len(resultats_ner)}")
print(f"Personnes trouvees   : {len(toutes_personnes)}")
print(f"Organisations        : {len(toutes_orgs)}")
print(f"Lieux geographiques  : {len(tous_lieux)}")
print()
print("Top 10 personnes :")
for p in list(toutes_personnes)[:10]:
    print(f"  {p}")
print()
print("Top 10 organisations :")
for o in list(toutes_orgs)[:10]:
    print(f"  {o}")

STATISTIQUES NER
Textes analyses      : 60
Personnes trouvees   : 90
Organisations        : 18
Lieux geographiques  : 19

Top 10 personnes :
  Jimmy Page
  Genres
  Philip Selway
  Queen
  Richard Wright
  Ian Stewart
  Amy Jade Winehouse
  Thom Yorke
  Dion
  Miles Davis

Top 10 organisations :
  Radiohead
  Alors
  Beyoncé
  Papaoutai and Formidable
  Daft Punk
  Group
  electro house
  The Rolling Stones
  Rolling Stone
  BE


### Sauvegarde

In [15]:
# Sauvegarde des textes bruts
with open(CHEMIN + "textes_crawles.json", "w", encoding="utf-8") as f:
    json.dump(tous_textes, f, indent=2, ensure_ascii=False)
print("Sauvegarde : textes_crawles.json")

# Sauvegarde des entites NER
with open(CHEMIN + "entites_ner.json", "w", encoding="utf-8") as f:
    json.dump(resultats_ner, f, indent=2, ensure_ascii=False)
print("Sauvegarde : entites_ner.json")

# Rapport de crawling
rapport = {
    "sources": ["Wikipedia API", "MusicBrainz API"],
    "artistes_cibles": len(ARTISTES),
    "textes_recuperes": len(tous_textes),
    "ethique": {
        "user_agent": "identifie",
        "delai_wikipedia": "1 seconde",
        "delai_musicbrainz": "1.5 secondes",
        "robots_txt": "respecte (APIs officielles utilisees)"
    },
    "ner": {
        "modele": "spacy en_core_web_sm",
        "personnes": len(toutes_personnes),
        "organisations": len(toutes_orgs),
        "lieux": len(tous_lieux)
    }
}
with open(CHEMIN + "rapport_crawling.json", "w", encoding="utf-8") as f:
    json.dump(rapport, f, indent=2, ensure_ascii=False)
print("Sauvegarde : rapport_crawling.json")

Sauvegarde : textes_crawles.json
Sauvegarde : entites_ner.json
Sauvegarde : rapport_crawling.json
